In [1]:
import networkx as nx
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm

# clustering
from sklearn.cluster import KMeans, DBSCAN, OPTICS

# deep stuff
import torch
from torch_geometric.utils import from_networkx
from torch_geometric.loader import DataLoader

/home/jann/GAT/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.community_results import community_metrics
from src.graph_based_clustering import label_propagation, louvain, leiden
from src.DGIModel import DGIModel
from src.tune import tune

# Load dataset

In [3]:
data_path = './data/deezer_clean_data/'

edges = os.path.join(data_path, 'HR_edges.csv')
genres = os.path.join(data_path, 'HR_genres.json')

In [4]:
edges_df = pd.read_csv(edges)
g = nx.from_pandas_edgelist(edges_df, source='node_1', target='node_2')

In [5]:
with open(genres, 'r') as f:
    genres_data = json.load(f)


In [6]:
def encode_genres(genre_list, unique_genres):
    encoding = [1 if genre in genre_list else 0 for genre in unique_genres]
    return encoding


def decode_genres(genre_vector, unique_genres):
    decoded_genres = [unique_genres[i] for i in range(len(genre_vector)) if genre_vector[i] == 1]
    return decoded_genres

In [7]:
all_genres = set()
for genres in genres_data.values():
    all_genres.update(genres)

all_genres = sorted(all_genres)

for user_id, genres in genres_data.items():
    if isinstance(user_id, str):
        user_id = int(user_id)
    if user_id in g.nodes:
        g.nodes[user_id]['genres'] = encode_genres(genres, all_genres)

In [8]:
sel_graph = 'deezer'
# nx.draw(g, with_labels=True)

# Apply graph based clustering

## Label Propagation

In [9]:
lp_labels = label_propagation(g)

# plot_communities(g, lp_labels, title=f"Label Propagation {sel_graph}")

# Calculate metrics for Louvain
lp_metrics = community_metrics(g, lp_labels)
lp_metrics

{'Number of Communities': 916,
 'Average Community Size': 59.577510917030565,
 'Modularity': 0.6737606352463761,
 'Coverage': 0.71361817094271,
 'Performance': 0.9460924050525683}

## Louvain

In [10]:
lv_labels = louvain(g)

# plot_communities(g, lv_labels, title=f"Louvain {sel_graph}")

# Calculate metrics for Louvain
lv_metrics = community_metrics(g, lv_labels)
lv_metrics

{'Number of Communities': 24,
 'Average Community Size': 2273.875,
 'Modularity': 0.7392975648922048,
 'Coverage': 0.8074275093235274,
 'Performance': 0.9270869907537564}

## Leiden

In [11]:
ld_labels = leiden(g)

# plot_communities(g, ld_labels, title=f"Leiden {sel_graph}")

# Calculate metrics for Louvain
ld_metrics = community_metrics(g, ld_labels)
ld_metrics

{'Number of Communities': 24,
 'Average Community Size': 2273.875,
 'Modularity': 0.7421684545988803,
 'Coverage': 0.8090995218806829,
 'Performance': 0.928518812822755}

# Apply GAT Embedding

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [13]:
data = from_networkx(g)
data.x = torch.tensor([g.nodes[n]['genres'] for n in g.nodes], dtype=torch.float)
data = data.to(device)

In [14]:
best_p = tune(data)
best_p

[I 2024-08-12 19:51:23,547] A new study created in memory with name: no-name-641a0e9c-6396-4529-a09c-229e88d0d53f
[I 2024-08-12 19:55:29,216] Trial 0 finished with value: 0.2274456456225456 and parameters: {'hidden_channels': 33, 'out_channels': 41, 'heads': 17, 'learning_rate': 0.0003181507588307312, 'dropout': 0.48145607890503117, 'weight_decay': 5.711907102730298e-05}. Best is trial 0 with value: 0.2274456456225456.
[I 2024-08-12 19:57:25,187] Trial 1 finished with value: 0.17055683175474842 and parameters: {'hidden_channels': 25, 'out_channels': 22, 'heads': 3, 'learning_rate': 0.0009977134025241678, 'dropout': 0.20727033345262455, 'weight_decay': 5.769776099356947e-05}. Best is trial 0 with value: 0.2274456456225456.
[I 2024-08-12 20:00:15,777] Trial 2 finished with value: 0.13904088072249468 and parameters: {'hidden_channels': 92, 'out_channels': 17, 'heads': 6, 'learning_rate': 0.009027603004342683, 'dropout': 0.1370444228936853, 'weight_decay': 4.774229453549677e-05}. Best is t

OutOfMemoryError: CUDA out of memory. Tried to allocate 8.38 GiB. GPU 

In [ ]:
best_p.best_params

In [ ]:
best_p.best_value

In [14]:
hidden_channels = 64
out_channels = 32
num_layers = 3
heads = 16
dropout = 0.15
lr = 1e-3
l2_reg = 5e-4
epochs = 100

In [15]:
train_loader = DataLoader([data], batch_size=1, shuffle=True)

In [16]:
model = DGIModel(in_channels=data.num_features, hidden_channels=hidden_channels, out_channels=out_channels, num_layers=num_layers, heads=heads, dropout=dropout)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=l2_reg)

accumulation_steps = 4

def train():
    for i, data in enumerate(train_loader):
        model.train()
        optimizer.zero_grad()
        pos_z, neg_z, summary = model(data)
        loss = model.dgi.loss(pos_z, neg_z, summary)
        loss = loss / accumulation_steps
        loss.backward()

        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

    return loss.item()


for epoch in tqdm(range(epochs)):
    loss = train()
    # if epoch % 10 == 0:
    print(f'Epoch {epoch}, Loss: {loss}')


  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:11<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.01 GiB. GPU 

In [ ]:
# Extract node embeddings
model.eval()
with torch.no_grad():
    node_embeddings = model.encoder(data.x, data.edge_index).detach().cpu().numpy()

In [ ]:
model_path = 'deezer.torch'
# torch.save(model.state_dict(), model_path)

# model = DGIModel(in_channels=data.num_features, hidden_channels=hidden_channels, out_channels=out_channels, num_layers=num_layers, heads=heads, dropout=dropout)
# model.load_state_dict(torch.load(model_path))
# model.eval()

In [ ]:
best_eps = 0
best_mod = -1

optics = OPTICS(min_samples=5)
communities = optics.fit_predict(node_embeddings)
# plot_communities(g, communities, title=f"GAT {sel_graph}")
community_metrics(g, dict(zip(range(g.number_of_nodes()), communities)))

{'Number of Communities': 857,
 'Average Community Size': 63.67911318553092,
 'Modularity': -0.00011071075613328701,
 'Coverage': 0.7942400873541254,
 'Performance': 0.2083772623359983}

In [ ]:
best_n = 950
best_mod = -1

kmeans = KMeans(n_clusters=best_n)
communities = kmeans.fit_predict(node_embeddings)

# plot_communities(g, communities, title=f"GAT {sel_graph}")
community_metrics(g, dict(zip(range(g.number_of_nodes()), communities)))

{'Number of Communities': 950,
 'Average Community Size': 57.445263157894736,
 'Modularity': -0.0001268534697850763,
 'Coverage': 0.0019570375068747215,
 'Performance': 0.9976134178971277}